# 02 · LLM Fundamentals

> **Source notes:** `LLMFundamentals.md`

What actually happens inside a language model? This notebook makes it concrete:
- **Tokenisation** — BPE in action with `tiktoken`
- **Sampling parameters** — temperature, top-p, top-k via a local Ollama SLM
- **Lost-in-the-middle** — empirical context position experiment
- **Training stages mental model** — prompt-level SFT vs RLHF behaviour differences

All LLM calls use **Ollama** (local, no API key needed).

## 0 · Environment Setup

```bash
# One-time: install Ollama and pull a model
winget install Ollama.Ollama        # Windows
# brew install ollama               # macOS
ollama pull phi3:mini               # ~2 GB download
```
Make sure `ollama serve` is running (or the desktop app is open) before executing cells.

In [ ]:
import tiktoken
import ollama

print("✓ Libraries imported")
print(f"  tiktoken version: {tiktoken.__version__}")
print(f"  Ollama client ready")

## 1 · Tokenisation — BPE in Action

The model never sees raw text. Text is broken into **tokens** (subword units) by a byte-pair encoding (BPE) vocabulary.

Key facts:
- ~1 token ≈ 0.75 English words
- Numbers tokenise byte-by-byte in some vocabs → arithmetic is hard
- The same string tokenises differently across model families

`tiktoken` is OpenAI's tokeniser library. `cl100k_base` is the vocabulary used by GPT-4 / GPT-3.5.

In [ ]:
# Simple tokenization example
text = "The pizza oven reaches 800°F."
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(text)

print(f"Text: {text}")
print(f"Token count: {len(tokens)}")
print(f"Token IDs: {tokens}")
print(f"Decoded tokens:")
for i, token_id in enumerate(tokens):
    print(f"  {i}: '{enc.decode([token_id])}'")

# Notice how 800 and °F are separate tokens

In [ ]:
import tiktoken

text = "gluten-free pepperoni with dairy-free mozzarella"

# GPT-3.5/4 tokenizer
enc_cl100k = tiktoken.get_encoding("cl100k_base")
tokens_cl100k = enc_cl100k.encode(text)

# GPT-4o tokenizer
enc_o200k = tiktoken.get_encoding("o200k_base")
tokens_o200k = enc_o200k.encode(text)

print(f"Text: {text}\n")
print(f"cl100k_base (GPT-3.5/4): {len(tokens_cl100k)} tokens")
print(f"  Tokens: {[enc_cl100k.decode([t]) for t in tokens_cl100k]}\n")
print(f"o200k_base (GPT-4o): {len(tokens_o200k)} tokens")
print(f"  Tokens: {[enc_o200k.decode([t]) for t in tokens_o200k]}\n")

# Cost comparison
cost_gpt35 = (len(tokens_cl100k) / 1000) * 0.002
cost_gpt4o = (len(tokens_o200k) / 1000) * 0.01
print(f"Cost for 1 encoding:")
print(f"  GPT-3.5: ${cost_gpt35:.6f}")
print(f"  GPT-4o:  ${cost_gpt4o:.6f}")
print(f"  Delta:   ${cost_gpt4o - cost_gpt35:.6f}")

## 2 · Sampling Parameters — Temperature, Top-p, Top-k

The model outputs a probability distribution over vocabulary at each step. Sampling parameters control how you draw the next token.

| Parameter | Effect |
|---|---|
| **Temperature = 0** | Argmax — always pick the most probable token. Deterministic, factual. |
| **Temperature = 1** | Sample from raw distribution. Creative, varied. |
| **Temperature > 1** | Flatten distribution → more random/chaotic. |
| **Top-p (nucleus)** | Only sample from smallest set of tokens summing to probability ≥ p. |
| **Top-k** | Only sample from the k most probable tokens. |

Below we ask the same factual question at different temperatures and observe consistency.

In [ ]:
import ollama

prompt = "Recommend a pizza"
temperatures = [0.0, 0.8, 1.5]

for temp in temperatures:
    print(f"\n{'='*60}")
    print(f"Temperature = {temp}")
    print('='*60)
    for i in range(5):
        response = ollama.generate(
            model='phi3:mini',
            prompt=prompt,
            options={'temperature': temp, 'num_predict': 50}
        )
        print(f"{i+1}. {response['response'][:100]}...")

## 3 · Lost-in-the-Middle — Context Position Matters

Models recall information from the **start** and **end** of long contexts more reliably than from the **middle**. This is the *lost-in-the-middle* effect (Liu et al., 2023).

We simulate it by embedding a target fact at different positions in a padded context and measuring retrieval accuracy.

In [ ]:
import ollama

# Create padded context with target fact at different positions
filler = "Here is some general information about pizza. " * 40  # ~2000 tokens
target = "IMPORTANT: The special today is Margherita at 50% off."
question = "What is the special today?"

positions = [0.0, 0.5, 1.0]  # start, middle, end
results = {pos: {'hits': 0, 'runs': 5} for pos in positions}

for position in positions:
    for run in range(5):
        # Inject target at position
        if position == 0.0:
            context = target + " " + filler
        elif position == 0.5:
            mid = len(filler) // 2
            context = filler[:mid] + " " + target + " " + filler[mid:]
        else:  # 1.0
            context = filler + " " + target

        full_prompt = f"{context}\n\nQuestion: {question}\nAnswer:"

        response = ollama.generate(
            model='phi3:mini',
            prompt=full_prompt,
            options={'temperature': 0.0, 'num_predict': 30}
        )

        if "Margherita" in response['response'] or "50%" in response['response']:
            results[position]['hits'] += 1

print("\nRecall by position:")
for pos in [0.0, 0.5, 1.0]:
    hit_rate = results[pos]['hits'] / results[pos]['runs'] * 100
    print(f"  Position {pos:.0%}: {results[pos]['hits']}/5 = {hit_rate:.0f}%")

## 4 · Training Stages Mental Model — SFT vs RLHF Behaviour

Three stages produce the assistant you call via API:

```
Stage 1: Pretraining   Raw transformer → learns language + world knowledge
Stage 2: SFT           Fine-tuned on (instruction, good response) pairs → follows instructions
Stage 3: RLHF / DPO   Aligned with human preferences → helpful, harmless, honest
```

We can probe the effects at the prompt level: a base-style completion prompt vs an instruction-following prompt shows how strongly the RLHF alignment suppresses raw continuation.

In [ ]:
import ollama

# Base-style: continuation prompt
base_prompt = "Customer: What sizes do you have?\nAI:"

# Instruct-style: chat message
response_base = ollama.generate(
    model='phi3:mini',
    prompt=base_prompt,
    options={'temperature': 0.0}
)

response_instruct = ollama.chat(
    model='phi3:mini',
    messages=[{'role': 'user', 'content': 'What sizes do you have?'}],
    options={'temperature': 0.0}
)

print("Base-style (continuation):")
print(response_base['response'][:200])
print("\n" + "="*60 + "\n")
print("Instruct-style (chat):")
print(response_instruct['message']['content'][:200])

## Summary

| Concept | Key Takeaway |
|---|---|
| Tokenisation | ~1 token ≈ 0.75 words; numbers and code are denser; use `tiktoken` for cost estimation |
| Temperature | 0 = deterministic factual; 0.7 = creative; >1 = noisy. Match to task. |
| Top-p / Top-k | Nucleus sampling limits the candidate pool; use with moderate temperature |
| Context position | Facts at start/end recalled better than middle — put critical context first |
| Training stages | Pretraining → SFT → RLHF produce progressively more useful and safe assistants |

**Next:** [PromptEngineering/notebook.ipynb](../PromptEngineering/notebook.ipynb) — build on this with reliable prompt patterns.